# KI ALIDOU - CHALLENGE BNP

# Importation

In [5]:

import pandas as pd
import numpy as np
import warnings


from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')


# Lecture des données

In [120]:
X_train = pd.read_csv('X_train.csv')
Y_train = pd.read_csv('Y_train.csv')
X_test = pd.read_csv('X_test.csv')

for col in set(X_train.columns) - set(X_test.columns):
    X_test[col] = np.nan
for col in set(X_test.columns) - set(X_train.columns):
    X_train[col] = np.nan

cols = sorted(X_train.columns)
X_train = X_train[cols]
X_test = X_test[cols]

train = X_train.merge(Y_train[['ID', 'fraud_flag']], on='ID', how='left')
print(f"Train : {train.shape}, Fraudes = {train['fraud_flag'].sum()} ({train['fraud_flag'].mean()*100:.2f}%)")
print(f"Test  : {X_test.shape}")

Train : (92790, 147), Fraudes = 1319 (1.42%)
Test  : (23198, 146)


# EDA

In [123]:
cols_prix = [f'cash_price{i}' for i in range(1, 25)]
cols_qte = [f'Nbr_of_prod_purchas{i}' for i in range(1, 25)]
cols_item = [f'item{i}' for i in range(1, 25)]
cols_make = [f'make{i}' for i in range(1, 25)]

## Taux de fraude par catégorie (item1)

In [127]:
cat_fraud = train.groupby('item1').agg(
    nb_obs=('fraud_flag', 'count'),
    nb_fraudes=('fraud_flag', 'sum'),
    taux_fraude=('fraud_flag', 'mean')
).sort_values('taux_fraude', ascending=False)
cat_fraud['taux_fraude_%'] = (cat_fraud['taux_fraude'] * 100).round(2)

In [129]:
cat_fraud[['nb_obs', 'nb_fraudes', 'taux_fraude_%']].head(20)

,nb_obs,nb_fraudes,taux_fraude_%
item1,,,
KITCHEN UTENSILS & GADGETS,3,1,33.33
HOME AND PERSONAL SECURITY,15,2,13.33
BABY CHANGING,17,2,11.76
WOMEN S CLOTHES,25,2,8.00
IMAGING EQUIPMENT,79,5,6.33
MEN S CLOTHES,18,1,5.56
FOOD PREPARATION,21,1,4.76
"TELEPHONES, FAX MACHINES & TWO-WAY RADIOS",2952,86,2.91
NURSERY FURNITURE,39,1,2.56


# Taux de fraude par item et fabriquant (tous les slots)

## Selon le fabricant

In [133]:
records = []
for i in range(1, 25):
    col_i = f'make{i}'
    mask = train[col_i].notna()
    if mask.sum() == 0:
        continue
    sub = train.loc[mask, [col_i, 'fraud_flag']].copy()
    sub.rename(columns={col_i: 'fabricant'}, inplace=True)
    records.append(sub)

all_items = pd.concat(records, ignore_index=True)
item_fraud = all_items.groupby('fabricant').agg(
    nb_apparitions=('fraud_flag', 'count'),
    nb_fraudes=('fraud_flag', 'sum'),
    taux_fraude=('fraud_flag', 'mean')
).sort_values('taux_fraude', ascending=False)
item_fraud['taux_fraude_%'] = (item_fraud['taux_fraude'] * 100).round(2)



In [137]:
item_fraud[item_fraud['nb_apparitions'] >= 30][['nb_apparitions', 'nb_fraudes', 'taux_fraude_%']].head(20)

,nb_apparitions,nb_fraudes,taux_fraude_%
fabricant,,,
OXO,64,10,15.62
ROBERT WELCH,47,5,10.64
LSA INTERNATIONAL,46,4,8.70
MAXI-COSI,125,9,7.20
LE CREUSET,208,12,5.77
DESIGN PROJECT BY RETAILER,38,2,5.26
SHNUGGLE,39,2,5.13
SILVER CROSS,154,7,4.55
PHILIPS,139,6,4.32


## Selon les items

In [140]:
records = []
for i in range(1, 25):
    col_i = f'item{i}'
    mask = train[col_i].notna()
    if mask.sum() == 0:
        continue
    sub = train.loc[mask, [col_i, 'fraud_flag']].copy()
    sub.rename(columns={col_i: 'item_cat'}, inplace=True)
    records.append(sub)

all_items = pd.concat(records, ignore_index=True)
item_fraud = all_items.groupby('item_cat').agg(
    nb_apparitions=('fraud_flag', 'count'),
    nb_fraudes=('fraud_flag', 'sum'),
    taux_fraude=('fraud_flag', 'mean')
).sort_values('taux_fraude', ascending=False)
item_fraud['taux_fraude_%'] = (item_fraud['taux_fraude'] * 100).round(2)



In [142]:
item_fraud[item_fraud['nb_apparitions'] >= 30][['nb_apparitions', 'nb_fraudes', 'taux_fraude_%']].head(20)

,nb_apparitions,nb_fraudes,taux_fraude_%
item_cat,,,
KITCHEN UTENSILS & GADGETS,31,11,35.48
STORAGE & ORGANISATION,39,8,20.51
LAUNDRY & CLOTHESCARE,34,5,14.71
IMAGING EQUIPMENT,85,7,8.24
MEN S CLOTHES,59,4,6.78
GAMING,36,2,5.56
AUDIO ACCESSORIES,1670,82,4.91
BARWARE,41,2,4.88
NURSERY ACCESSORIES,111,5,4.50


# Taux de fraude selon la diversité

## Selon le nombre de catégorie

In [146]:
train['nb_cat_uniques'] = train[cols_item].apply(lambda x: x.dropna().nunique(), axis=1)
train['nb_fab_uniques'] = train[cols_make].apply(lambda x: x.dropna().nunique(), axis=1)

div_cat = train.groupby('nb_cat_uniques').agg(
    nb_obs=('fraud_flag', 'count'),
    nb_fraudes=('fraud_flag', 'sum'),
    taux_fraude=('fraud_flag', 'mean')
)
div_cat['taux_fraude_%'] = (div_cat['taux_fraude'] * 100).round(2)

In [147]:
div_cat[['nb_obs', 'nb_fraudes', 'taux_fraude_%']]

,nb_obs,nb_fraudes,taux_fraude_%
nb_cat_uniques,,,
1,51078,671,1.31
2,32299,567,1.76
3,7184,56,0.78
4,1429,10,0.70
5,372,5,1.34
6,167,5,2.99
7,84,0,0.00
8,61,1,1.64
9,42,0,0.00


## Selon le nombre de fabricant

In [149]:
div_fab = train.groupby('nb_fab_uniques').agg(
    nb_obs=('fraud_flag', 'count'),
    nb_fraudes=('fraud_flag', 'sum'),
    taux_fraude=('fraud_flag', 'mean')
)
div_fab['taux_fraude_%'] = (div_fab['taux_fraude'] * 100).round(2)

In [150]:
div_fab[['nb_obs', 'nb_fraudes', 'taux_fraude_%']]

,nb_obs,nb_fraudes,taux_fraude_%
nb_fab_uniques,,,
0,685,7,1.02
1,57206,737,1.29
2,32566,545,1.67
3,1499,15,1.00
4,381,3,0.79
5,158,5,3.16
6,103,1,0.97
7,52,1,1.92
8,43,1,2.33


## Taux de fraude selon le prix max du panier

In [152]:
train['prix_max'] = train[cols_prix].max(axis=1)

bins_prix = [0, 100, 300, 500, 800, 1000, 1500, 2000, 3000, 100000]
labels_prix = ['0-100', '100-300', '300-500', '500-800', '800-1000', '1000-1500', '1500-2000', '2000-3000', '3000+']
train['tranche_prix_max'] = pd.cut(train['prix_max'], bins=bins_prix, labels=labels_prix)

prix_fraud = train.groupby('tranche_prix_max', observed=False).agg(
    nb_obs=('fraud_flag', 'count'),
    nb_fraudes=('fraud_flag', 'sum'),
    taux_fraude=('fraud_flag', 'mean')
)
prix_fraud['taux_fraude_%'] = (prix_fraud['taux_fraude'] * 100).round(2)

In [153]:
prix_fraud[['nb_obs', 'nb_fraudes', 'taux_fraude_%']]

,nb_obs,nb_fraudes,taux_fraude_%
tranche_prix_max,,,
0-100,3,0,0.00
100-300,459,8,1.74
300-500,13738,38,0.28
500-800,14030,120,0.86
800-1000,20350,225,1.11
1000-1500,25302,459,1.81
1500-2000,10576,263,2.49
2000-3000,7110,186,2.62
3000+,1222,20,1.64


# STATISTIQUES DES VARIABLES QUANTITATIVES (PRIX & QUANTITÉS)

In [155]:
train['prix_total'] = train[cols_prix].sum(axis=1)
train['nb_produits'] = train[cols_qte].sum(axis=1)

quant_vars = {
    'cash_price1': train['cash_price1'],
    'prix_max': train['prix_max'],
    'prix_total': train['prix_total'],
    'Nb_of_items': train['Nb_of_items'],
    'nb_produits': train['nb_produits'],
    'Nbr_of_prod_purchas1': train['Nbr_of_prod_purchas1'],
}

In [156]:
print(f"\n{'Variable':<25} {'Groupe':<12} {'Moyenne':>10} {'Médiane':>10} {'Écart-type':>12} {'Min':>8} {'Max':>10}")
print("-" * 100)
for var_name, series in quant_vars.items():
    for label, flag in [('Non fraude', 0), ('Fraude', 1)]:
        s = series[train['fraud_flag'] == flag].dropna()
        print(f"{var_name:<25} {label:<12} {s.mean():>10.2f} {s.median():>10.2f} {s.std():>12.2f} {s.min():>8.2f} {s.max():>10.2f}")
    print()


Variable                  Groupe          Moyenne    Médiane   Écart-type      Min        Max
----------------------------------------------------------------------------------------------------
cash_price1               Non fraude      1089.48     949.00       709.75     2.00   21995.00
cash_price1               Fraude          1400.98    1199.00       742.29     8.00    6999.00

prix_max                  Non fraude      1136.24     999.00       692.24    79.00   21995.00
prix_max                  Fraude          1433.65    1249.00       725.03   195.00    6999.00

prix_total                Non fraude      1230.25    1099.00       769.67   219.00   21995.00
prix_total                Fraude          1547.47    1379.00       831.02   305.00    9018.00

Nb_of_items               Non fraude         1.76       1.00         1.46     1.00      60.00
Nb_of_items               Fraude             1.76       2.00         1.55     1.00      24.00

nb_produits               Non fraude         1.8

# Nettoyage colonnes temporaires EDA

In [158]:

for c in ['tranche_prix_max', 'nb_cat_uniques', 'nb_fab_uniques', 'prix_max', 'prix_total', 'nb_produits']:
    if c in train.columns:
        train.drop(columns=[c], inplace=True)

In [170]:
train.shape

(92790, 147)

# FEATURE ENGINEERING

In [173]:
def creer_features(df):
    r = df.copy()
    c_prix = [f'cash_price{i}' for i in range(1, 25)]
    c_qte = [f'Nbr_of_prod_purchas{i}' for i in range(1, 25)]
    c_item = [f'item{i}' for i in range(1, 25)]
    c_make = [f'make{i}' for i in range(1, 25)]

    r['prix_total'] = r[c_prix].sum(axis=1)
    r['prix_moyen'] = r[c_prix].mean(axis=1)
    r['prix_max'] = r[c_prix].max(axis=1)
    r['prix_min'] = r[c_prix].apply(lambda x: x[x > 0].min() if (x > 0).any() else 0, axis=1)
    r['prix_std'] = r[c_prix].std(axis=1).fillna(0)
    r['log_prix_total'] = np.log1p(r['prix_total'].clip(0))
    r['log_prix1'] = np.log1p(r['cash_price1'].fillna(0).clip(0))

    r['ratio_prix1_total'] = r['cash_price1'].fillna(0) / r['prix_total'].replace(0, 1)
    r['ratio_max_total'] = r['prix_max'] / r['prix_total'].replace(0, 1)

    r['nb_produits'] = r[c_qte].sum(axis=1)
    r['qte_max'] = r[c_qte].max(axis=1)
    r['log_nb_produits'] = np.log1p(r['nb_produits'])

    r['prix_par_produit'] = r['prix_total'] / r['nb_produits'].replace(0, 1)
    r['prix_par_item'] = r['prix_total'] / r['Nb_of_items'].replace(0, 1)
    r['produits_par_item'] = r['nb_produits'] / r['Nb_of_items'].replace(0, 1)
    r['log_prix_par_produit'] = np.log1p(r['prix_par_produit'].clip(0))

    r['nb_slots_prix'] = (r[c_prix] > 0).sum(axis=1)
    r['ratio_slots_items'] = r['nb_slots_prix'] / r['Nb_of_items'].replace(0, 1)
    r['diff_slots_items'] = r['nb_slots_prix'] - r['Nb_of_items']

    r['est_mono_item'] = (r['Nb_of_items'] == 1).astype(int)
    r['a_produit_cher'] = (r['prix_max'] > 1500).astype(int)

    risk_cats = ['COMPUTERS', 'TELEPHONES, FAX MACHINES & TWO-WAY RADIOS']
    r['item1_risque'] = r['item1'].isin(risk_cats).astype(int)
    r['est_apple'] = (r['make1'] == 'APPLE').astype(int)

    r['nb_fab_uniques'] = r[c_make].apply(lambda x: x.dropna().nunique(), axis=1)
    r['nb_cat_uniques'] = r[c_item].apply(lambda x: x.dropna().nunique(), axis=1)
    r['ratio_diversite_fab'] = r['nb_fab_uniques'] / r['Nb_of_items'].replace(0, 1)
    r['tous_meme_fab'] = (r['nb_fab_uniques'] == 1).astype(int)

    r['apple_x_cher'] = r['est_apple'] * r['a_produit_cher']
    r['risque_x_mono'] = r['item1_risque'] * r['est_mono_item']
    r['valeur_slot1'] = r['cash_price1'].fillna(0) * r['Nbr_of_prod_purchas1'].fillna(0)

    r['tranche_prix_5'] = pd.cut(r['prix_total'], bins=[0, 200, 500, 1000, 2000, 1e6],
                                  labels=[0, 1, 2, 3, 4]).astype(float).fillna(0)

    r['item1_make1'] = r['item1'].fillna('UNK') + '_' + r['make1'].fillna('UNK')
    r['item1_item2'] = r['item1'].fillna('UNK') + '_' + r['item2'].fillna('SEUL')
    r['make1_make2'] = r['make1'].fillna('UNK') + '_' + r['make2'].fillna('SEUL')
    r['item1_tranche'] = r['item1'].fillna('UNK') + '_' + r['tranche_prix_5'].astype(int).astype(str)

    return r

# Création des features

In [176]:
train = creer_features(train)
X_test = creer_features(X_test)


# ENCODAGE CATÉGORIEL

In [183]:
cols_encoder = ['item1', 'make1', 'model1', 'goods_code1', 'item2', 'make2', 'model2',
                'item3', 'make3', 'item1_make1', 'item1_item2', 'make1_make2', 'item1_tranche']
cols_encoder = [c for c in cols_encoder if c in train.columns]

taux_global = train['fraud_flag'].mean()

for col in cols_encoder:
    mean_target = train.groupby(col)['fraud_flag'].mean()
    count = train.groupby(col)['fraud_flag'].count()
    smooth = 50 if train[col].nunique() > 1000 else 20
    smoothed = (count * mean_target + smooth * taux_global) / (count + smooth)
    train[f'{col}_taux'] = train[col].map(smoothed).fillna(taux_global)
    X_test[f'{col}_taux'] = X_test[col].map(smoothed).fillna(taux_global)

    freq = train[col].value_counts(normalize=True)
    train[f'{col}_freq'] = train[col].map(freq).fillna(0)
    X_test[f'{col}_freq'] = X_test[col].map(freq).fillna(0)

    cnt = train[col].value_counts()
    train[f'{col}_count'] = train[col].map(cnt).fillna(0)
    X_test[f'{col}_count'] = X_test[col].map(cnt).fillna(0)

    train[f'{col}_dev'] = train[f'{col}_taux'] - taux_global
    X_test[f'{col}_dev'] = X_test[f'{col}_taux'] - taux_global


# PRÉPARATION FINALE

In [188]:
ids_test = X_test['ID'].values

exclure = ['ID', 'fraud_flag'] + \
          [f'item{i}' for i in range(1, 25)] + \
          [f'make{i}' for i in range(1, 25)] + \
          [f'model{i}' for i in range(1, 25)] + \
          [f'goods_code{i}' for i in range(1, 25)] + \
          ['item1_make1', 'item1_item2', 'make1_make2', 'item1_tranche']

features = [c for c in train.columns if c not in exclure and c in X_test.columns]

X = train[features].copy()
y = train['fraud_flag'].copy()
X_t = X_test[features].copy()

for col in X.columns:
    if X[col].dtype == 'object' or X_t[col].dtype == 'object':
        le = LabelEncoder()
        all_vals = pd.concat([X[col].astype(str), X_t[col].astype(str)])
        le.fit(all_vals)
        X[col] = le.transform(X[col].astype(str))
        X_t[col] = le.transform(X_t[col].astype(str))

X = X.fillna(-999).astype(float)
X_t = X_t.fillna(-999).astype(float)

poids_pos = (y == 0).sum() / (y == 1).sum()
print(f"Features : {X.shape[1]}, Poids classe positive : {poids_pos:.1f}")

Features : 132, Poids classe positive : 69.3


# ENTRAÎNEMENT FINAL

## Configuration des modèles

In [192]:
modeles_config = {
    'GradientBoosting': lambda: GradientBoostingClassifier(
        n_estimators=800, learning_rate=0.03, max_depth=5,
        min_samples_leaf=50, subsample=0.7, random_state=123
    ),
    'XGBoost': lambda: xgb.XGBClassifier(
        n_estimators=526, learning_rate=0.02, max_depth=6,
        min_child_weight=50, subsample=0.7, colsample_bytree=0.6,
        reg_alpha=0.5, reg_lambda=2.0, scale_pos_weight=poids_pos,
        random_state=777, n_jobs=-1, eval_metric='logloss'
    ),
    'HistGBM': lambda: HistGradientBoostingClassifier(
        max_iter=800, learning_rate=0.02, max_depth=5,
        min_samples_leaf=50, l2_regularization=2.0,
        max_bins=200, class_weight='balanced', random_state=314
    ),
}



best_blend = {'GradientBoosting': 0.25, 'XGBoost': 0.10, 'HistGBM': 0.65}

In [197]:


pred_finale = np.zeros(len(X_t))

for nom, model_fn in modeles_config.items():
    w = best_blend[nom]
    print(f"\n--- {nom} (poids={w}) ---")

    model = model_fn()
    model.fit(X, y)

 
    # Prédiction sur le test
    proba_test = model.predict_proba(X_t)[:, 1]
    pred_finale += w * proba_test

  
    print(f"  Test  : min={proba_test.min():.4f}, max={proba_test.max():.4f}, moy={proba_test.mean():.4f}")


--- GradientBoosting (poids=0.25) ---
  Test  : min=0.0001, max=0.9849, moy=0.0130

--- XGBoost (poids=0.1) ---
  Test  : min=0.0002, max=0.9909, moy=0.1996

--- HistGBM (poids=0.65) ---
  Test  : min=0.0004, max=0.9877, moy=0.2251


# Résultat Prédiction finale

In [236]:
res = pd.DataFrame({'Min': [pred_finale.min()], 'Max': [pred_finale.max()], 'Moy': [pred_finale.mean()]})

In [238]:
res

,Min,Max,Moy
0,0.000402,0.983978,0.169502


# GÉNÉRATION DE LA SOUMISSION

In [244]:
sub = pd.DataFrame({'ID': ids_test, 'fraud_flag': pred_finale})
sub.to_csv('KI_Alidou_Y_test.csv', index=True ,index_label='index')
print(f"Fichier sauvegardé : KI_Alidou_Y_test.csv ({len(sub)} lignes)")

Fichier sauvegardé : KI_Alidou_Y_test.csv (23198 lignes)
